# Homework #2: Basic Reliability & Hazard Analysis

Assignment due March 10, 2026 for AREN 8940: Infrastructure & Community Resilience

In [12]:
import pandas as pd
import geopandas as gpd
import folium

from scipy.stats import norm
import numpy as np
import math
import sympy as sp

## Task 2: Poisson Distribution and Probability of Occurance

This [link](https://earthquake.usgs.gov/earthquakes/map/?extent=-89.97343,-607.5&extent=89.97277,967.5&range=search&timeZone=utc&search=%7B%22name%22:%22Search%20Results%22,%22params%22:%7B%22starttime%22:%221900-01-01%2000:00:00%22,%22endtime%22:%222022-12-30%2023:59:59%22,%22minmagnitude%22:8,%22orderby%22:%22time%22%7D%7D) from the United States Geological Survey (USGS) shows all magnitude 8 (or larger) earthquakes that have been detected on the planet from the year 1900 until the end of the year 2022. Download the dataset by scrolling down in the "search results" part of the website or see the CSV file in the same folder as this notebook - it is used for the questions in this section.

2-1) Make a time history plot of the number of events per year.

2-2) Determine the average number of earthquakes per year (# of earthquakes/number of observed years).

2-3) Compute the probability of having a year without a single magnitude 8 (or larger) earthquake.

2-4) Compute the probability of 2 magnitude 8 (or larger) earthquakes in a single year.

2-5) Look at the year 2007. Based on the data, how unlikely was that year?

## Task 3: USGS Seismic Hazard Database

3-1) Using the USGS database presented in class, determine the following values for Salt Lake City, UT, at locations with soil class B/C.

In [ ]:
PGA = peak ground acceleration

3-2) Using the USGS database presented in class, determine the following values for Charleston, SC, at locations with soil class B/C.

3-3) Using the USGS database presented in class, determine the following values for San Francisco, CA, at locations with soil class B/C.

## Task 4: Working with IN-CORE Data Service

This task uses the shelby-police-fire-stations.csv file posted to Canvas (and uploaded to this folder), which provides the fire station building inventory for Shelby County, TN.

3-1) Import the CSV file into this Jupyter notebook and create a local dataset of building inventory in IN-CORE from the CSV provided.

In [4]:
# Pull CSV file into a dataframe from where it's hosted on the github website. USE RAW DATA, not the plain website URL.
shelby_fire = pd.read_csv('https://raw.githubusercontent.com/julia-ehlers/AREN-8940-infra-comm-resilience/refs/heads/main/homework-2/shelby_police_fire_stations.csv')

# Test to make sure dataframe is working appropriately.
shelby_fire.head(10)

,parid,struct_typ,year_built,no_stories,a_stories,b_stories,bsmt_type,occ_type,appr_bldg,cont_val,...,nstra_cst,nstrd_cst,dgn_lvl,archetype,appr_land,ffe_elev,g_elev,lhsm_elev,lon,lat
0,076036 00018_1,PC2,1959,1,NaN,NaN,NaN,GOV2,61000.0,91500.0,...,NaN,NaN,NaN,11,13600.0,NaN,NaN,NaN,-90.059260,35.030940
1,C0244 00197_1,URM,1908,1,NaN,NaN,NaN,GOV2,57600.0,86400.0,...,NaN,NaN,NaN,11,18550.0,NaN,NaN,NaN,-89.663860,35.043370
2,C0232 00230_1,PC2,1953,1,NaN,NaN,NaN,GOV2,158200.0,237300.0,...,NaN,NaN,NaN,11,0.0,NaN,NaN,NaN,-89.731020,35.071820
3,NaN,S1,1972,2,NaN,NaN,NaN,GOV2,NaN,NaN,...,NaN,NaN,NaN,11,NaN,NaN,NaN,NaN,-89.949520,35.204240
4,NaN,URM,1974,1,NaN,NaN,NaN,GOV2,NaN,NaN,...,NaN,NaN,NaN,11,NaN,NaN,NaN,NaN,-89.921710,35.037040
5,NaN,URM,1940,2,NaN,NaN,NaN,GOV2,NaN,NaN,...,NaN,NaN,NaN,11,NaN,NaN,NaN,NaN,-89.960527,35.074203
6,NaN,URM,1971,1,NaN,NaN,NaN,GOV2,NaN,NaN,...,NaN,NaN,NaN,11,NaN,NaN,NaN,NaN,-89.960527,35.074203
7,NaN,URM,1965,1,NaN,NaN,NaN,GOV2,NaN,NaN,...,NaN,NaN,NaN,11,NaN,NaN,NaN,NaN,-89.960527,35.074203
8,NaN,S3,1982,2,NaN,NaN,NaN,GOV2,NaN,NaN,...,NaN,NaN,NaN,11,NaN,NaN,NaN,NaN,-89.960527,35.074203
9,NaN,URM,1958,1,NaN,NaN,NaN,GOV2,NaN,NaN,...,NaN,NaN,NaN,11,NaN,NaN,NaN,NaN,-90.028630,35.216730


3-2) Display the building inventory on the interactive map by using Geopandas (refer to the example solved in class) based on the following columns:

* year-built
* no-stories
* occ-type

Comment on the distribution of these structural characteristics.

In [ ]:
# Convert the dataframe from above into a geodataframe using the latitude and longitude.
shelby_gdf = gpd.GeoDataFrame(
    shelby_fire,
    geometry = gpd.points_from_xy(shelby_fire['lon'], shelby_fire['lat']),
    crs = 'EPSG:4326'
)

# Create a blank map to put the markers on.
shelby_map = folium.Map(location = [shelby_gdf.geometry.y.mean(), shelby_gdf.geometry.x.mean()], zoom_start = 10)

# Add markers with year built, number of stories, and occupancy type information.
for i, row in shelby_gdf.iterrows():
    # Text formatting for popups:
    popup_text = f"""
    <b>Year Built:</b> {row['year_built']} <br>
    <b>Stories:</b> {row['no_stories']} <br>
    <b>Occupancy Type:</b> {row['occ_type']}
    """

    # Adding actual markers for each building:
    folium.Marker(
        location = [row.geometry.y, row.geometry.x],
        popup = folium.Popup(popup_text, max_width = 200)
    ).add_to(shelby_map)

# Save map as interactive html file.
shelby_map.save('shelby_map.html')

# NOTE: If you are generating the map for yourself, uncomment the code below to determine where the file is saved on your computer/device.
# If you do not want to run the code, the HTML file has been uploaded to this Github repo - this can be downloaded and opened in browser.
# import os
# print(os.getcwd())

## Task 5: Working with IN-CORE Hazard Service

Using pyincore-viz package, visualize the tornado with the ID: 0a44ae8605f0462bd4263ac available on IN-CORE hazard service.

## Task 6: Reliability Analysis using Monte Carlo Simulation

Consider the following limit state equation for a reinforced concrete beam:

$$
g = A_S \cdot F_y \cdot (d - 0.59 \cdot \frac{A_S \cdot F_y}{f'_c \cdot b}) - [D + L]
$$

The parameters of the random variables in this equation are listed in the following table.

| Parameter | Mean | COV | Distribution |
| --- | --- | --- | --- |
| $f'_c$ | 3750 psi | 12.5% | Normal |
| $F_y$ | 42.5 ksi | 11.5% | Lognormal |
| $D$ | 100 k-ft | 10.5% | Normal |
| $L$ | 200 k-ft | 17.5% | Lognormal |

The other parameters can be treated as deterministic variables. Their values are as follows: $b = 15 in$, $d = 24 in$, $A_S = 8 in^2$.

Use Monte Carlo simulation to plot the predicted probability of failure versus the number of samples. Starting with a small number of samples, increase the number of samples until the estimated failure probability is on the order of $10^{-4}$.

Note that D and L are in units of kip-ft in the table above, and have been multiplied by 12,000 in the code below to be converted to lb-in, so the output g is in lb-in.

In [11]:
# DEFINITIONS

def lognormal_char_calcs(mu, COV):
    # Function finds mu_ln and sigma_ln from mu and COV to generate a lognormal distribution.

    sigma_ln = math.sqrt(math.log(1 + COV**2))
    mu_ln = math.log(mu) - (0.5 * (sigma_ln**2))

    return mu_ln, sigma_ln

def random_var_generation(ns, mu_fc, COV_fc, mu_Fy, COV_Fy, mu_D, COV_D, mu_L, COV_L):
    # Function generates normal distributions for f'_c and D, and lognormal distributions for F_y and L.
    # NOTE: variable ns is the number of samples in the distribution

    # Find standard deviations for f'_c and D.
    sigma_fc = mu_fc * COV_fc
    sigma_D = mu_D * COV_D

    # Find lognormal values for F_y and L.
    mu_ln_Fy, sigma_ln_Fy = lognormal_char_calcs(mu_Fy, COV_Fy)
    mu_ln_L, sigma_ln_L = lognormal_char_calcs(mu_L, COV_L)

    # Create distributions for all four random variables.
    fc_dist = np.random.normal(loc = mu_fc, scale = sigma_fc, size = ns)
    Fy_dist = np.random.lognormal(mean = mu_ln_Fy, sigma = sigma_ln_Fy, size = ns)
    D_dist = np.random.normal(loc = mu_D, scale = sigma_D, size = ns)
    L_dist = np.random.lognormal(mean = mu_ln_L, sigma = sigma_ln_L, size = ns)

    return fc_dist, Fy_dist, D_dist, L_dist

def calculate_prob_failure(fc_dist, Fy_dist, D_dist, L_dist):
    # Calculates g - the probability of a reinforced concrete beam failing.

    # Define empty array for the values of g.
    g_dist = []

    # Establish deterministic variables.
    b = 15      # Units: inches
    d = 24      # Units: inches
    As = 8      # Units: square inches

    # Solve for g using distributions and deterministic variables.
    g_dist = (As * Fy_dist * (d - (0.59 * ((As * Fy_dist) / (fc_dist * b))))) - (D_dist + L_dist)
    
    return g_dist

def monte_carlo_simulation(ns, mu_fc, COV_fc, mu_Fy, COV_Fy, mu_D, COV_D, mu_L, COV_L):
    # Runs a Monte Carlo simulation for a set sample size. Outputs sample size, mean and standard deviation.

    # Define distributions of the right size.
    fc_dist, Fy_dist, D_dist, L_dist = random_var_generation(ns, mu_fc, COV_fc, mu_Fy, COV_Fy, mu_D, COV_D, mu_L, COV_L)

    # Find the probability of failure (a distribution for g) from the random variables.
    g_dist = calculate_prob_failure(fc_dist, Fy_dist, D_dist, L_dist)

    # Find mean and standard deviation of the distribution of g values.
    mu_g = np.mean(g_dist)
    sigma_g = np.mean(g_dist)

    return ns, mu_g, sigma_g


In [ ]:
# MAIN BODY OF CODE

# Define deterministic variables
b = 15      # Units: inches
d = 24      # Units: inches
A_s = 8     # Units: square inches

# Define random variable means, COVs, and standard deviations.
fc_mean = 3750                          # Units: pounds per square inch
fc_COV = 0.125                          # Units: multiply by 100 for %
fc_stddev = fc_mean * fc_COV            # Units: pounds per square inch

Fy_mean = 42500                         # Units: pounds per square inch
Fy_COV = 0.115                          # Units: multiply by 100 for %
Fy_stddev = Fy_mean * Fy_COV            # Units: pounds per square inch

D_mean = 100 * 12000                    # Units: pound-inches
D_COV = 0.105                           # Units: multiply by 100 for %
D_stddev = D_mean * D_COV               # Units: pound-inches

L_mean = 200 * 12000                    # Units: pound-inches
L_COV = 0.175                           # Units: multiply by 100 for %
L_stddev = L_mean * L_COV               # Units: pound-inches

# Find actual mean value for g.
mu_g_actual = (A_s * Fy_mean * (d - (0.59 * ((A_s * Fy_mean) / (fc_mean * b))))) - (D_mean + L_mean)

# Find actual standard deviation of g using ERROR PROPAGATION.
# Define symbolic variables.
fc = sp.symbols('fc')
Fy = sp.symbols('Fy')
D = sp.symbols('D')
L = sp.symbols('L')
g = (A_s * Fy * (d - (0.59 * ((A_s * Fy) / (fc * b))))) - (D + L)

# Find partial derivatives of g.
dg_dfc = sp.diff(g, fc)
dg_dFy = sp.diff(g, Fy)
dg_dD = sp.diff(g, D)
dg_dL = sp.diff(g, L)

# Plug in actual values for function.
var_means = {fc: fc_mean, Fy: Fy_mean, D: D_mean, L: L_mean}
var_sigmas = [fc_stddev, Fy_stddev, D_stddev, L_stddev]
partials = [dg_dfc, dg_dFy, dg_dD, dg_dL]

# Use error propagation to find standard deviation.
sigma_g_actual = sp.sqrt(sum(
    (p.subs(var_means))**2 * s**2
    for p, s in zip(partials, var_sigmas)
))

# Define sample sizes to test.
sample_nums = np.linspace(start = 10000, stop = 10000000, retstep = 10000)

# Define deterministic variables.
b = 15      # Units: inches
d = 24      # Units: inches
A_s = 8     # Units: square inches

# Define random variable constraints.
fc_mean = 3750          # Units: pounds per square inch
fc_COV = 0.125          # Units: multiply by 100 for %
Fy_mean = 42500         # Units: pounds per square inch
Fy_COV = 0.115          # Units: multiply by 100 for %
D_mean = 100 * 12000    # Units: pound-inches
D_COV = 0.105           # Units: multiply by 100 for %
L_mean = 200 * 12000    # Units: pound-inches
L_COV = 0.175           # Units: multiply by 100 for %

# Monte Carlo Simulation
for i in sample_nums:
    random_var_generation(ns = i, )


"""
# Number of samples used for each distribution.
num_samples = 1000

# Define deterministic variables.
b = 15      # Units: inches
d = 24      # Units: inches
A_s = 8     # Units: square inches

# Define normal distribution for f'_c.
f_c_mean = 3750                         # Units: pounds per square inch
f_c_COV = 0.125                         # Units: multiply by 100 for %
f_c_stddev = f_c_mean * f_c_COV         # Units: pounds per square inch
f_c_dist = np.random.normal(loc = f_c_mean, scale = f_c_stddev, size = num_samples)

# Define lognormal distribution for F_y.
F_y_mean = 42500                        # Units: pounds per square inch
F_y_COV = 0.115                         # Units: multiply by 100 for %
F_y_stddev = F_y_mean * F_y_COV         # Units: pounds per square inch
F_y_ln_stddev = math.sqrt(math.log(1 + F_y_COV**2))
F_y_ln_mean = math.log(F_y_mean) - (0.5 * (F_y_ln_stddev**2))

# Define normal distribution for D.
D_mean = 100 * 12000                    # Units: pound-inches
D_COV = 0.105                           # Units: multiply by 100 for %
D_stddev = D_mean * D_COV               # Units: pound-inches

# Define lognormal distribution for L.
L_mean = 200 * 12000                    # Units: pound-inches
L_COV = 0.175                           # Units: multiply by 100 for %
L_stddev = L_mean * L_COV               # Units: pound-inches
L_ln_stddev = math.sqrt(math.log(1 + L_COV**2))
L_ln_mean = math.log(L_mean) - (0.5 * (L_ln_stddev**2))

# Check to make sure that distributions have worked appropriately.
# print("f'_c distribution:", np.mean(f_c_dist))
# print("F_y distribution:", np.mean(F_y_dist))
# print("D distribution:", np.mean(D_dist))
# print("L distribution:", np.mean(L_dist))
"""